# Name: Keshab Bhattarai StudentID: 240453
# Exploratory Data Analysis (EDA) Using PySpark and Spark SQL

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

In [2]:
spark = SparkSession.builder.appName("Data Analysis Supermarket Sales").master("local[*]").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/07 12:12:43 WARN Utils: Your hostname, Keshabs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.12.8.199 instead (on interface en0)
26/06/07 12:12:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/07 12:12:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/07 12:12:44 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


# Loading and Inspection

In [3]:
df_infer = spark.read.csv("supermarket_sales.csv", header=True, inferSchema=True)
df_infer.printSchema()
df_infer.show(10)

root
 |-- transaction_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- total_sales: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)

+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+
|transaction_id|      date|customer_id|product_category|product_name|quantity|unit_price|total_sales|payment_method|      city|
+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+
|         T0001|2025-02-05|       C027|       Beverages|      Coffee|       5|     13.38|       66.9|       eWallet|Biratnagar|
|         T0002|2025-02-25|       C060|       Beverages|  S

# Null Value Handling

In [5]:
# Null values in each column
null_counts = df_infer.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_infer.columns
])

print("Null values in each column:")
null_counts.show()

Null values in each column:
+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+
|transaction_id|date|customer_id|product_category|product_name|quantity|unit_price|total_sales|payment_method|city|
+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+
|             0|   0|          0|               0|           0|       0|         0|          0|             0|   0|
+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+



In [6]:
# Duplicate records
total_rows = df_infer.count()
unique_rows = df_infer.dropDuplicates().count()
duplicate_count = total_rows - unique_rows

print("Total rows:", total_rows)
print("Unique rows:", unique_rows)
print("Duplicate records:", duplicate_count)

Total rows: 500
Unique rows: 500
Duplicate records: 0


In [7]:
# Optional: Display duplicate rows if any
duplicate_rows = (
    df_infer
    .groupBy(df_infer.columns)
    .count()
    .filter(F.col("count") > 1)
)

duplicate_rows.show(truncate=False)

+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+-----+
|transaction_id|date|customer_id|product_category|product_name|quantity|unit_price|total_sales|payment_method|city|count|
+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+-----+
+--------------+----+-----------+----------------+------------+--------+----------+-----------+--------------+----+-----+



# Descriptive Stats

In [9]:
# Automatically detect numerical columns
from pyspark.sql.types import IntegerType, DoubleType, FloatType, LongType
numeric_cols = [
    field.name
    for field in df_infer.schema.fields
    if isinstance(field.dataType, (IntegerType, DoubleType, FloatType, LongType))
]

print("Numerical columns:", numeric_cols)

# Descriptive statistics
df_infer.select(numeric_cols).describe().show()

Numerical columns: ['quantity', 'unit_price', 'total_sales']
+-------+------------------+------------------+-----------------+
|summary|          quantity|        unit_price|      total_sales|
+-------+------------------+------------------+-----------------+
|  count|               500|               500|              500|
|   mean|             5.524|25.628980000000013|        145.03284|
| stddev|2.9730627649628962|13.991613046066618|120.2808106872146|
|    min|                 1|              1.73|             1.73|
|    max|                10|             49.98|            498.1|
+-------+------------------+------------------+-----------------+



# EDA

In [10]:
total_revenue = (
    df_infer
    .agg(F.sum("total_sales").alias("total_supermarket_revenue"))
)

total_revenue.show()

+-------------------------+
|total_supermarket_revenue|
+-------------------------+
|                 72516.42|
+-------------------------+



In [11]:
top_5_categories = (
    df_infer
    .groupBy("product_category")
    .agg(F.sum("total_sales").alias("total_category_sales"))
    .orderBy(F.desc("total_category_sales"))
    .limit(5)
)

top_5_categories.show(truncate=False)

+----------------+--------------------+
|product_category|total_category_sales|
+----------------+--------------------+
|Snacks          |15938.829999999998  |
|Dairy           |15883.500000000004  |
|Bakery          |14209.069999999994  |
|Beverages       |14186.970000000005  |
|Produce         |12298.050000000001  |
+----------------+--------------------+



In [12]:
average_values = (
    df_infer
    .agg(
        F.avg("quantity").alias("average_quantity_sold"),
        F.avg("unit_price").alias("average_unit_price"),
        F.avg("total_sales").alias("average_sales_amount")
    )
)

average_values.show()

+---------------------+------------------+--------------------+
|average_quantity_sold|average_unit_price|average_sales_amount|
+---------------------+------------------+--------------------+
|                5.524|25.628980000000013|           145.03284|
+---------------------+------------------+--------------------+



In [13]:
highest_revenue_city = (
    df_infer
    .groupBy("city")
    .agg(F.sum("total_sales").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
    .limit(1)
)

highest_revenue_city.show(truncate=False)

+---------+------------------+
|city     |total_revenue     |
+---------+------------------+
|Bhaktapur|16650.069999999996|
+---------+------------------+



In [14]:
most_frequent_payment_method = (
    df_infer
    .groupBy("payment_method")
    .agg(F.count("*").alias("transaction_count"))
    .orderBy(F.desc("transaction_count"))
    .limit(1)
)

most_frequent_payment_method.show(truncate=False)

+--------------+-----------------+
|payment_method|transaction_count|
+--------------+-----------------+
|Card          |177              |
+--------------+-----------------+



In [15]:
sales_by_category = (
    df_infer
    .groupBy("product_category")
    .agg(F.sum("total_sales").alias("total_sales"))
    .orderBy(F.desc("total_sales"))
)

sales_by_category.show(truncate=False)

+----------------+------------------+
|product_category|total_sales       |
+----------------+------------------+
|Snacks          |15938.829999999998|
|Dairy           |15883.500000000004|
|Bakery          |14209.069999999994|
|Beverages       |14186.970000000005|
|Produce         |12298.050000000001|
+----------------+------------------+



In [16]:
top_10_transactions = (
    df_infer
    .orderBy(F.desc("total_sales"))
    .limit(10)
)

top_10_transactions.show(truncate=False)

+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+
|transaction_id|date      |customer_id|product_category|product_name|quantity|unit_price|total_sales|payment_method|city      |
+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+
|T0015         |2025-05-16|C065       |Snacks          |Crackers    |10      |49.81     |498.1      |eWallet       |Kathmandu |
|T0093         |2025-05-15|C073       |Snacks          |Chocolate   |10      |48.87     |488.7      |Cash          |Pokhara   |
|T0100         |2025-01-26|C054       |Bakery          |Bread       |10      |48.07     |480.7      |eWallet       |Pokhara   |
|T0161         |2025-05-24|C085       |Dairy           |Milk        |10      |47.9      |479.0      |eWallet       |Kathmandu |
|T0098         |2025-01-22|C064       |Dairy           |Cheese      |10      |47.53     |475.3      |Cas

In [17]:
df_sales_level = (
    df_infer
    .withColumn(
        "sales_level",
        F.when(F.col("total_sales") > 100, "High")
         .when((F.col("total_sales") >= 50) & (F.col("total_sales") <= 100), "Medium")
         .otherwise("Low")
    )
)

df_sales_level.show(10, truncate=False)

+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+-----------+
|transaction_id|date      |customer_id|product_category|product_name|quantity|unit_price|total_sales|payment_method|city      |sales_level|
+--------------+----------+-----------+----------------+------------+--------+----------+-----------+--------------+----------+-----------+
|T0001         |2025-02-05|C027       |Beverages       |Coffee      |5       |13.38     |66.9       |eWallet       |Biratnagar|Medium     |
|T0002         |2025-02-25|C060       |Beverages       |Soft Drink  |1       |2.95      |2.95       |eWallet       |Biratnagar|Low        |
|T0003         |2025-04-25|C072       |Beverages       |Tea         |9       |21.85     |196.65     |Cash          |Pokhara   |High       |
|T0004         |2025-03-28|C027       |Snacks          |Chocolate   |5       |9.04      |45.2       |Cash          |Bhaktapur |Low        |
|T0005         |2025

In [18]:
sales_level_count = (
    df_sales_level
    .groupBy("sales_level")
    .agg(F.count("*").alias("transaction_count"))
    .orderBy(F.desc("transaction_count"))
)

sales_level_count.show()

+-----------+-----------------+
|sales_level|transaction_count|
+-----------+-----------------+
|       High|              261|
|        Low|              139|
|     Medium|              100|
+-----------+-----------------+

